# Simple RNN, LSTM, GRU for Sentiment Analysis

In [1]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install tensorflow-datasets

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --upgrade jupyter ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Libraries
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences  
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout

In [2]:
# import the dataset
dataset, info = tfds.load("imdb_reviews" , split = ["train", "test"], as_supervised = True, with_info = True)

In [3]:
print(info)

tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_dir='C:\\Users\\Prikshit_Ishi\\tensorflow_datasets\\imdb_reviews\\plain_text\\1.0.0',
    file_format=tfrecord,
    download_size=Unknown size,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
    supervised_keys=('text', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
      

In [6]:
dataset

[<_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>,
 <_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>]

In [7]:
print(type(dataset))

<class 'list'>


In [8]:
print(type(dataset[0]))   # training data

<class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>


In [9]:
train_data = dataset[0].as_numpy_iterator()

In [10]:
train_data

NumpyIterator(iterator=<tensorflow.python.data.ops.iterator_ops.OwnedIterator object at 0x00000213AED36F90>)

In [14]:
for i in range(3):
    review, label = next(train_data)
    print(f"REVIEW: {review.decode('utf-8')},\n LABEL : {"Positive" if label == 1 else "Negative"}")

REVIEW: Sure, this one isn't really a blockbuster, nor does it target such a position. "Dieter" is the first name of a quite popular German musician, who is either loved or hated for his kind of acting and thats exactly what this movie is about. It is based on the autobiography "Dieter Bohlen" wrote a few years ago but isn't meant to be accurate on that. The movie is filled with some sexual offensive content (at least for American standard) which is either amusing (not for the other "actors" of course) or dumb - it depends on your individual kind of humor or on you being a "Bohlen"-Fan or not. Technically speaking there isn't much to criticize. Speaking of me I find this movie to be an OK-movie.,
 LABEL : Negative
REVIEW: During a sleepless night, I was switching through the channels & found this embarrassment of a movie. What were they thinking?<br /><br />If this is life after "Remote Control" for Kari (Wuhrer) Salin, no wonder she's gone nowhere.<br /><br />And why did David Keith t

In [15]:
train_data, train_labels = [], []
test_data, test_labels = [], []

In [16]:
for review, label in dataset[0]:
    train_data.append(review.numpy().decode('utf-8'))
    train_labels.append(label.numpy())

In [26]:
train_labels[0]

np.int64(0)

In [27]:
for review, label in dataset[1]:
    test_data.append(review.numpy().decode('utf-8'))
    test_labels.append(label.numpy())

In [50]:
train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

In [51]:
train_labels[0]

np.int64(0)

In [52]:
# Tokenization
VOCAB_SIZE = 10000   # Most frequent vocab
tokenizer = Tokenizer(num_words = VOCAB_SIZE, oov_token = "<OOV>")
tokenizer.fit_on_texts(train_data)

In [53]:
# converting the text into numbers
# test to sequences
train_sequences = tokenizer.texts_to_sequences(train_data)
test_sequences = tokenizer.texts_to_sequences(test_data)

In [54]:
train_sequences[0]

[12,
 14,
 33,
 425,
 392,
 18,
 90,
 28,
 1,
 9,
 32,
 1366,
 3585,
 40,
 486,
 1,
 197,
 24,
 85,
 154,
 19,
 12,
 213,
 329,
 28,
 66,
 247,
 215,
 9,
 477,
 58,
 66,
 85,
 114,
 98,
 22,
 5675,
 12,
 1322,
 643,
 767,
 12,
 18,
 7,
 33,
 400,
 8170,
 176,
 2455,
 416,
 2,
 89,
 1231,
 137,
 69,
 146,
 52,
 2,
 1,
 7577,
 69,
 229,
 66,
 2933,
 16,
 1,
 2904,
 1,
 1,
 1479,
 4940,
 3,
 39,
 3900,
 117,
 1584,
 17,
 3585,
 14,
 162,
 19,
 4,
 1231,
 917,
 7917,
 9,
 4,
 18,
 13,
 14,
 4139,
 5,
 99,
 145,
 1214,
 11,
 242,
 683,
 13,
 48,
 24,
 100,
 38,
 12,
 7181,
 5515,
 38,
 1366,
 1,
 50,
 401,
 11,
 98,
 1197,
 867,
 141,
 10]

In [55]:
len(train_data[0])

709

In [56]:
len(train_data[1])

617

In [57]:
max_len = 200

train_padded = pad_sequences(train_sequences, maxlen = max_len, padding = 'post', truncating = "post")
test_padded = pad_sequences(test_sequences, maxlen = max_len, padding = 'post', truncating = "post")

In [58]:
len(train_padded[1])

200

In [59]:
# Machine Learning algo

In [60]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression()

log_model.fit(train_padded, train_labels)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [61]:
log_model.score(test_padded, test_labels)

0.51436

In [62]:
# Naive bayes
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(train_padded, train_labels)
nb_model.score(test_padded, test_labels)

0.51

In [63]:
len(train_padded)

25000

In [64]:
len(train_labels)

25000

# Use Simple RNN for Sentiment analysis

In [69]:
rnn_model = Sequential([
                            Embedding(VOCAB_SIZE, 64),
                            SimpleRNN(64, return_sequences = True),
                            Dropout(0.2),
                            SimpleRNN(32),
                            Dense(32, activation = "relu"),
                            Dense(1, activation = "sigmoid")
])

In [70]:
rnn_model.compile(loss = "binary_crossentropy", optimizer = "adam", metrics = ['accuracy'])

In [ ]:
histroy_rnn = rnn_model.fit(train_padded, train_labels, epochs = 3, verbose = 1)

Epoch 1/3
374/782 ━━━━━━━━━━━━━━━━━━━━ 36s 90ms/step - accuracy: 0.5022 - loss: 0.7015 